# Окупаемость аренды - задача B (классификация)

Отобьет ли точка аренду.окупаемость = выручка / аренда. Если окупаемость ниже медианы по train - точка в зоне риска.

In [ ]:
!pip install catboost shap

## Загрузка и таргет

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, recall_score, precision_score

In [ ]:
RANDOM_STATE = 67

In [ ]:
train = pd.read_csv("train.csv")
val = pd.read_csv("val.csv")
train.columns = train.columns.str.strip()
val.columns = val.columns.str.strip()

In [ ]:
train = train[train["RENT_HEX"] > 0].copy()
val = val[val["RENT_HEX"] > 0].copy()

In [ ]:
train.shape, val.shape

## Разбиение, метка и препроцессинг

In [ ]:
target = "target_2"
num_cols = (train.select_dtypes("number")
            .drop(columns=["ID", "target_1", "target_2", "RENT_HEX"], errors="ignore")
            .columns.tolist())
cols = ["REGION", "CITY", "subject"]
ratio = train[target] / train["RENT_HEX"]

In [ ]:
X_train_raw, X_test_raw, r_train, r_test = train_test_split(train, ratio, test_size=0.2, random_state=RANDOM_STATE)

In [ ]:
thr = r_train.median()
y_train = (r_train < thr).astype(int)
y_test = (r_test  < thr).astype(int)
y_val = ((val[target] / val["RENT_HEX"]) < thr).astype(int)

In [ ]:
medians = X_train_raw[num_cols].median()
X_train = X_train_raw[num_cols].fillna(medians).copy()
X_test  = X_test_raw[num_cols].fillna(medians).copy()
X_val   = val[num_cols].fillna(medians).copy()

In [ ]:
te = TargetEncoder(random_state=RANDOM_STATE)
X_train[cols] = te.fit_transform(X_train_raw[cols], y_train)
X_test[cols]  = te.transform(X_test_raw[cols])
X_val[cols]   = te.transform(val[cols])

In [ ]:
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)
X_val_scaled   = scaler.transform(X_val)

Порог окупаемости

In [ ]:
round(thr, 1)

Доля "не окупает"

In [ ]:
print("train:", round(y_train.mean(), 3), "test:", round(y_test.mean(), 3), "val:", round(y_val.mean(), 3))

## Сравнение 6 классификаторов

In [ ]:
pos_w = (y_train == 0).sum() / (y_train == 1).sum()

models = {
 "LogReg": (LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE), "scaled"),
 "KNN": (KNeighborsClassifier(n_neighbors=25), "scaled"),
 "DecisionTree": (DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=RANDOM_STATE), "raw"),
 "RandomForest": (RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1), "raw"),
 "CatBoost": (CatBoostClassifier(iterations=400, depth=6, learning_rate=0.05, scale_pos_weight=pos_w,
                                     random_state=RANDOM_STATE, verbose=0, allow_writing_files=False), "raw"),
 "MLP": (MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=RANDOM_STATE), "scaled"),
}
sets = {"scaled": (X_train_scaled, X_test_scaled, X_val_scaled),
        "raw":    (X_train.values, X_test.values, X_val.values)}

In [ ]:
def metrics(model, Xte, Xva):
    p_test = model.predict_proba(Xte)[:, 1]
    p_val  = model.predict_proba(Xva)[:, 1]
    return {
        "AUC_test":  roc_auc_score(y_test, p_test),
        "Rec_test":  recall_score(y_test,  p_test >= 0.5),
        "Prec_test": precision_score(y_test, p_test >= 0.5, zero_division=0),
        "AUC_val":   roc_auc_score(y_val, p_val),
        "Rec_val":   recall_score(y_val,  p_val >= 0.5),
        "Prec_val":  precision_score(y_val, p_val >= 0.5, zero_division=0),
    }

rows = []

logreg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
logreg.fit(X_train_scaled, y_train)
rows.append({"model": "LogReg", **metrics(logreg, X_test_scaled, X_val_scaled)})

knn = KNeighborsClassifier(n_neighbors=25)
knn.fit(X_train_scaled, y_train)
rows.append({"model": "KNN", **metrics(knn, X_test_scaled, X_val_scaled)})

tree = DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=RANDOM_STATE)
tree.fit(X_train.values, y_train)
rows.append({"model": "DecisionTree", **metrics(tree, X_test.values, X_val.values)})

forest = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
forest.fit(X_train.values, y_train)
rows.append({"model": "RandomForest", **metrics(forest, X_test.values, X_val.values)})

cat = CatBoostClassifier(iterations=400, depth=6, learning_rate=0.05, scale_pos_weight=pos_w,
                         random_state=RANDOM_STATE, verbose=0, allow_writing_files=False)
cat.fit(X_train.values, y_train)
rows.append({"model": "CatBoost", **metrics(cat, X_test.values, X_val.values)})

mlp = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=RANDOM_STATE)
mlp.fit(X_train_scaled, y_train)
rows.append({"model": "MLP", **metrics(mlp, X_test_scaled, X_val_scaled)})

res_B = pd.DataFrame(rows).sort_values("AUC_val", ascending=False).round(3)
res_B

Здесь ROC-AUC = 0.8. Модели реально предсказывают окупаемость по гео-признакам.
CatBoost ожидаемо лучший. На val метрики ниже.

In [ ]:
best = res_B.sort_values("AUC_val", ascending=False).iloc[0]["model"]
best

In [ ]:
res_B.plot(x="model", y=["AUC_test", "AUC_val"], kind="bar", figsize=(10, 4))
plt.axhline(0.5, ls="--", color="gray")
plt.title("Задача B: ROC-AUC test vs val (окупаемость)")
plt.ylabel("ROC-AUC")
plt.show()

## SHAP

Смотрим, на какие признаки опирается CatBoost

In [ ]:
import shap

sample = X_test.iloc[:500]
explainer = shap.TreeExplainer(cat)      #
shap_values = explainer.shap_values(sample)

In [ ]:
shap.summary_plot(shap_values, sample, plot_type="bar", show=False, max_display=12)
plt.show()